# Seoul Station Stacking V20 No-Peak

Seoul (`RKSI`) adaptation of the KDAL V20-aligned station-stacking workflow. The notebook uses the existing Asia 11 AM parquet contract, Fahrenheit-native modeling, Wunderground-only settlement highs, and an expanding 2022–2025 validation design.


In [ ]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Skipping features without any observed values.*")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "calibration" / "asia_station_stacking.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing src/calibration/asia_station_stacking.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CITY_ID = "seoul"
CITY_LABEL = "Seoul"
STATION_ID = "RKSI"
TIMEZONE = "Asia/Seoul"
DATA_ROOT = PROJECT_ROOT / "data" / "calibration" / "asia_11am"
OUTPUT_DIR = DATA_ROOT / "models" / f"v20_{CITY_ID}_no_peak"
PROVIDERS = ("gfs", "gefs", "jma_msm")
TIMING_MODE = "asia_same_day_11am_live_safe"
FEATURE_VERSION = "v20_asia_no_peak"
TRAINING_PROFILE = "v20_aligned"
TARGET_SOURCE = "wunderground_only"
TARGET_MODE = "remaining_warmup"
OPTUNA_TRIALS = 30
STACK_OPTUNA_TRIALS = 30
OPTUNA_STARTUP_TRIALS = 15
STACK_OPTUNA_STARTUP_TRIALS = 15
FAST_MODE = False
EXPORT_MODEL_WEIGHTS = True
MODEL_VERSION = f"station_high_regressor_v20_{CITY_ID}_no_peak_stack"
PROJECT_ROOT


In [ ]:
import numpy as np
import pandas as pd

from src.calibration.asia_station_stacking import (
    ASIA_PROVIDERS,
    ASIA_TEST_YEAR,
    ASIA_TIMING_MODE,
    asia_expanding_folds,
    build_asia_station_wide_dataset,
    provider_readiness,
)
from src.calibration.station_stacking import (
    StationStackingConfig,
    V20_ASIA_NO_PEAK_FEATURE_VERSION,
    missing_model_dependencies,
    run_station_year_split_experiment,
)
from src.export_station_stacking_v2_models import export_station_model_weights


## City contract

- Existing Asia parquet data rooted at `data/calibration/asia_11am`
- Local 11 AM live-safe observation cutoff
- GFS, GEFS, and JMA MSM forecast inputs
- Wunderground-only daily settlement high target
- Fahrenheit-native model values with Celsius reporting


In [ ]:
fold_spec = pd.DataFrame(
    [
        {
            "fold": fold.name,
            "train_start_year": fold.train_start_year,
            "train_end_year": fold.train_end_year,
            "validation_year": fold.validation_year,
            "validation_weight": 1.0,
        }
        for fold in asia_expanding_folds()
    ]
)
fold_spec


## Provider readiness


In [ ]:
readiness = provider_readiness(DATA_ROOT, CITY_ID, providers=PROVIDERS)
readiness


## Build the live-safe modeling frame


In [ ]:
features = build_asia_station_wide_dataset(
    DATA_ROOT,
    CITY_ID,
    feature_version=FEATURE_VERSION,
    providers=PROVIDERS,
)
features[[
    "contract_date",
    "actual_high_f",
    "observed_high_temp_through_as_of_f",
    "gfs_high_f",
    "gefs_high_f",
    "jma_msm_high_f",
    "strict_quality_ok",
]].head()


In [ ]:
feature_coverage = (
    features[["actual_high_f", "observed_high_temp_through_as_of_f", *[f"{p}_high_f" for p in PROVIDERS]]]
    .notna()
    .mean()
    .rename("non_null_fraction")
    .to_frame()
)
feature_coverage


## Train and score


In [ ]:
missing_packages = missing_model_dependencies(("xgboost", "lightgbm", "catboost", "optuna"))
if missing_packages:
    raise ImportError(
        "Missing station-stacking ML packages: "
        + ", ".join(missing_packages)
        + ". Install them with: python -m pip install -r requirements.txt"
    )

config = StationStackingConfig(
    station_id=STATION_ID,
    project_root=PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
    fast_mode=FAST_MODE,
    optuna_trials=OPTUNA_TRIALS,
    stack_optuna_trials=STACK_OPTUNA_TRIALS,
    optuna_startup_trials=OPTUNA_STARTUP_TRIALS,
    stack_optuna_startup_trials=STACK_OPTUNA_STARTUP_TRIALS,
    optuna_metric="mae_f",
    feature_version=FEATURE_VERSION,
    training_profile=TRAINING_PROFILE,
    target_mode=TARGET_MODE,
    target_source=TARGET_SOURCE,
    max_feature_missing_fraction=0.03,
    base_model_methods=("xgboost", "lightgbm", "catboost"),
    stack_enabled=True,
    hyperparameter_space="wide",
    year_split_folds=asia_expanding_folds(),
    year_split_validation_weights={2023: 1.0, 2024: 1.0, 2025: 1.0},
    year_split_test_train_years=(2022, 2025),
    year_split_test_year=ASIA_TEST_YEAR,
    output_dir=OUTPUT_DIR,
    prebuilt_features=features,
)
config.resolved_optuna_storage_path()


In [ ]:
result = run_station_year_split_experiment(config)
result.scoreboard


## Celsius reporting and export


In [ ]:
celsius_predictions = result.test_predictions.copy()
for column in ("actual_high_f", "predicted_high_f", "error_f"):
    if column in celsius_predictions:
        celsius_predictions[column.replace("_f", "_c")] = pd.to_numeric(celsius_predictions[column], errors="coerce") * 5.0 / 9.0
celsius_predictions.head()


In [ ]:
if EXPORT_MODEL_WEIGHTS:
    exported_weights = export_station_model_weights(
        project_root=PROJECT_ROOT,
        station_id=STATION_ID,
        city_id=CITY_ID,
        artifact_dir=config.resolved_output_dir(),
        model_version=MODEL_VERSION,
        timing_mode=config.timing_mode,
        providers=tuple(config.providers),
        feature_version=config.effective_feature_version,
        training_profile=config.effective_training_profile,
        optuna_metric=config.effective_optuna_metric,
        target_mode=config.effective_target_mode,
        target_source=config.effective_target_source,
        base_model_methods=tuple(config.effective_base_model_methods),
        stack_enabled=config.stack_enabled,
        source_pipeline=f"notebooks/station_stacking_v20_asia_no_peak/{CITY_ID}",
    )
    exported_weights.bundle_path, exported_weights.manifest_path
else:
    print("Model export disabled for this notebook.")


In [ ]:
result.output_paths
